# Phase 2 — Exploratory Data Analysis

**Goal:** Understand distributions, trends, and relationships in the master dataset before modeling.

Key outputs:
- Correlation matrix (staffing vs outcomes)
- Time series per country
- Regional comparison (North / South / East / West Europe)
- Pre vs post COVID comparison
- Missing data heatmap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROCESSED = '../data/processed/'
FIGURES = '../figures/'

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 150

df = pd.read_csv(PROCESSED + 'master_dataset.csv')
print('Shape:', df.shape)
df.head()

## 1. Missing Data Overview

In [ ]:
missing = df.isnull().mean().sort_values(ascending=False) * 100
print(missing[missing > 0].round(1).to_string())

## 2. Descriptive Statistics by Region

In [ ]:
df.groupby('region')[['nurses_per_10k','mortality_ami_30d','mortality_stroke_30d','avg_length_of_stay']].mean().round(2)

## 3. Correlation Matrix

In [ ]:
cols = ['nurses_per_10k','mortality_ami_30d','mortality_stroke_30d','avg_length_of_stay','gdp_per_capita','health_exp_pct_gdp','beds_per_1000']
corr = df[cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Matrix — Key Variables')
plt.tight_layout()
plt.savefig(FIGURES + 'correlation_matrix.png')
plt.show()

## 4. Nurse Staffing Over Time by Region

In [ ]:
region_year = df.groupby(['year','region'])['nurses_per_10k'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
for region, grp in region_year.groupby('region'):
    ax.plot(grp['year'], grp['nurses_per_10k'], marker='o', label=region)
ax.axvspan(2020, 2022, alpha=0.1, color='red', label='COVID period')
ax.set_xlabel('Year')
ax.set_ylabel('Nurses per 10,000 population')
ax.set_title('Nurse Staffing Trends by European Region')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES + 'staffing_trends_by_region.png')
plt.show()

## 5. Nurse Staffing vs AMI Mortality (Scatter)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, outcome, label in zip(axes, ['mortality_ami_30d','mortality_stroke_30d'], ['30-day AMI Mortality (%)','30-day Stroke Mortality (%)']):
    sns.scatterplot(data=df, x='nurses_per_10k', y=outcome, hue='region', ax=ax, alpha=0.6)
    ax.set_xlabel('Nurses per 10,000')
    ax.set_ylabel(label)
    ax.set_title(f'Staffing vs {label}')

plt.tight_layout()
plt.savefig(FIGURES + 'staffing_vs_outcomes_scatter.png')
plt.show()

## 6. Pre vs Post COVID Comparison

In [ ]:
covid_compare = df.groupby('covid_period')[['nurses_per_10k','mortality_ami_30d','mortality_stroke_30d','avg_length_of_stay']].mean().round(2)
print(covid_compare)